In [1]:
import math
import numpy as np
import copy

X = "X"
O = "O"
EMPTY = None

def initial_state():
    return [[EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY]]

def player(board):
    count = 0
    for row in board:
        for cell in row:
            if cell:
                count += 1
    if count % 2 != 0:
        return O
    return X

def actions(board):
    res = set()
    for i in range(3):
        for j in range(3):
            if board[i][j] == EMPTY:
                res.add((i, j))
    return res

def result(board, action):
    result_board = copy.deepcopy(board)
    i, j = action
    result_board[i][j] = player(board)
    return result_board

def winner(board):
    for i in range(3):
        if board[i][0] == board[i][1] == board[i][2] and board[i][0] is not None:
            return board[i][0]
        if board[0][i] == board[1][i] == board[2][i] and board[0][i] is not None:
            return board[0][i]
            
    if board[0][0] == board[1][1] == board[2][2] and board[0][0] is not None:
        return board[0][0]
    if board[0][2] == board[1][1] == board[2][0] and board[0][2] is not None:
        return board[0][2]
    return None

def terminal(board):
    if winner(board) is not None:
        return True
    for row in board:
        if EMPTY in row:
            return False
    return True

def utility(board):
    winner_val = winner(board)
    if winner_val == X:
        return 1
    elif winner_val == O:
        return -1
    return 0

def maxValue(state):
    if terminal(state):
        return utility(state)
    v = -math.inf
    for action in actions(state):
        v = max(v, minValue(result(state, action)))
    return v

def minValue(state):
    if terminal(state):
        return utility(state)
    v = math.inf
    for action in actions(state):
        v = min(v, maxValue(result(state, action)))
    return v

def minimax(board):
    current_player = player(board)
    if current_player == X:
        v = -math.inf
        move = None
        for action in actions(board):
            check = minValue(result(board, action))
            if check > v:
                v = check
                move = action
    else:
        v = math.inf
        move = None
        for action in actions(board):
            check = maxValue(result(board, action))
            if check < v:
                v = check
                move = action
    return move

if __name__ == "__main__":
    board = initial_state()
    user = input("Chọn quân (X/O): ").upper()
    if user == X:
        ai = O
    else:
        ai = X
        
    while not terminal(board):
        curr_p = player(board)
        if curr_p == user:
            print("\nLượt của bạn:")
            try:
                i = int(input("Hàng (0-2): "))
                j = int(input("Cột (0-2): "))
                if board[i][j] == EMPTY:
                    board = result(board, (i, j))
                else:
                    print("Ô đã chọn, thử lại.")
                    continue
            except:
                print("Lỗi nhập liệu.")
                continue
        else:
            print("\nMáy đang đi...")
            move = minimax(board)
            board = result(board, move)
        
        print(np.array(board))

    res_winner = winner(board)
    if res_winner is None:
        print("\nHòa!")
    else:
        print(f"\n{res_winner} Thắng!")


Lượt của bạn:
[[None None None]
 [None 'X' None]
 [None None None]]

Máy đang đi...
[['O' None None]
 [None 'X' None]
 [None None None]]

Lượt của bạn:
[['O' None None]
 [None 'X' None]
 [None None 'X']]

Máy đang đi...
[['O' None None]
 [None 'X' None]
 ['O' None 'X']]

Lượt của bạn:
[['O' None None]
 [None 'X' 'X']
 ['O' None 'X']]

Máy đang đi...
[['O' None None]
 ['O' 'X' 'X']
 ['O' None 'X']]

O Thắng!


In [4]:
import math
import copy

X = "X"
O = "O"
EMPTY = None

def initial_state(n):
    return [[EMPTY for _ in range(n)] for _ in range(n)]

def player(board):
    count = 0
    for row in board:
        for cell in row:
            if cell:
                count += 1
    return O if count % 2 != 0 else X

def actions(board):
    res = set()
    for i in range(len(board)):
        for j in range(len(board)):
            if board[i][j] == EMPTY:
                res.add((i, j))
    return res

def result(board, action):
    new_board = copy.deepcopy(board)
    i, j = action
    new_board[i][j] = player(board)
    return new_board

def winner(board):
    n = len(board)
    for i in range(n):
        if all(board[i][j] == board[i][0] != EMPTY for j in range(n)):
            return board[i][0]
        if all(board[j][i] == board[0][i] != EMPTY for j in range(n)):
            return board[0][i]
    if all(board[i][i] == board[0][0] != EMPTY for i in range(n)):
        return board[0][0]
    if all(board[i][n-1-i] == board[0][n-1] != EMPTY for i in range(n)):
        return board[0][n-1]
    return None

def terminal(board):
    if winner(board) is not None:
        return True
    return all(all(cell != EMPTY for cell in row) for row in board)

def heuristic_evaluation(board):
    res = winner(board)
    if res == X: return 1000
    if res == O: return -1000
    return 0

def minimax_alpha_beta(board, depth, alpha, beta, is_maximizing):
    if terminal(board) or depth == 0:
        return heuristic_evaluation(board)
    
    if is_maximizing:
        v = -math.inf
        for action in actions(board):
            v = max(v, minimax_alpha_beta(result(board, action), depth - 1, alpha, beta, False))
            alpha = max(alpha, v)
            if beta <= alpha:
                break
        return v
    else:
        v = math.inf
        for action in actions(board):
            v = min(v, minimax_alpha_beta(result(board, action), depth - 1, alpha, beta, True))
            beta = min(beta, v)
            if beta <= alpha:
                break
        return v

def get_best_move(board, depth_limit=4):
    curr_player = player(board)
    best_move = None
    
    if curr_player == X:
        v = -math.inf
        for action in actions(board):
            score = minimax_alpha_beta(result(board, action), depth_limit - 1, -math.inf, math.inf, False)
            if score > v:
                v = score
                best_move = action
    else:
        v = math.inf
        for action in actions(board):
            score = minimax_alpha_beta(result(board, action), depth_limit - 1, -math.inf, math.inf, True)
            if score < v:
                v = score
                best_move = action
    return best_move

if __name__ == "__main__":
    n = int(input("Nhập kích thước ma trận N: "))
    board = initial_state(n)
    
    while not terminal(board):
        curr_p = player(board)
        if curr_p == X:
            print(f"\nLượt người chơi {X}:")
            r, c = map(int, input("Nhập hàng và cột (vd: 0 1): ").split())
            if board[r][c] == EMPTY:
                board = result(board, (r, c))
            else: continue
        else:
            print("\nMáy đang tính toán...")
            # Giới hạn độ sâu để đảm bảo tốc độ xử lý cho ma trận lớn
            move = get_best_move(board, depth_limit=3 if n > 3 else 9)
            board = result(board, move)
        
        for row in board:
            print(row)

    res = winner(board)
    print(f"Kết quả: {res if res else 'Hòa'}")


Lượt người chơi X:
[None, None, None]
[None, 'X', None]
[None, None, None]

Máy đang tính toán...
['O', None, None]
[None, 'X', None]
[None, None, None]

Lượt người chơi X:
['O', None, None]
[None, 'X', 'X']
[None, None, None]

Máy đang tính toán...
['O', None, None]
['O', 'X', 'X']
[None, None, None]

Lượt người chơi X:
['O', None, None]
['O', 'X', 'X']
['X', None, None]

Máy đang tính toán...
['O', None, 'O']
['O', 'X', 'X']
['X', None, None]

Lượt người chơi X:
['O', 'X', 'O']
['O', 'X', 'X']
['X', None, None]

Máy đang tính toán...
['O', 'X', 'O']
['O', 'X', 'X']
['X', 'O', None]

Lượt người chơi X:
['O', 'X', 'O']
['O', 'X', 'X']
['X', 'O', 'X']
Kết quả: Hòa
